# 👁️ RetinAI-DR: 8-Stage Experimental Progression & Final Model Training
### Senior AI/ML Engineer Production Pipeline

This notebook implements the systematic 8-stage experimental progression for Diabetic Retinopathy detection:
1. **Baseline** (Raw 224px, unweighted CE, uniform sampling)
2. **Exp 1: Data Quality Gate** (Blur filter, illumination check, landmark verification)
3. **Exp 2: Preprocessing** (Retina circular crop, Ben Graham local color enhancement, 384px)
4. **Exp 3: Targeted Augmentation** (Dihedral D4 group symmetry, RandAugment, Color Jitter)
5. **Exp 4: Class Imbalance Solution** (Dynamic minority data augmentation + BalancedBatchSampler + Focal Loss)
6. **Exp 5: Backbone Scaling** (EfficientNet-B4 + ConvNeXt-Tiny)
7. **Exp 6: Progressive Fine-Tuning** (2-stage head warmup, LLRD, Cosine Annealing, 4-fold TTA)
8. **Exp 7: Multi-Backbone Ensemble & Cohen's Kappa Threshold Optimization** (Nelder-Mead cutoffs)

**Target Metrics:** Accuracy > 90%, QWK > 0.912.

> **Prerequisite:** Set runtime to GPU via `Runtime` -> `Change runtime type` -> select `T4 GPU`.

In [ ]:
# Step 1: Environment & GPU Verification
import torch
print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
!nvidia-smi

In [ ]:
# Step 2: Install Dependencies
!pip install -q timm albumentations scikit-learn pandas opencv-python Pillow pyarrow scipy reportlab
import os, sys
sys.path.append(os.path.abspath("src"))

In [ ]:
# Step 3: Fetch Dataset (APTOS 2019 from Hugging Face)
import os, urllib.request
os.makedirs("data/raw/hf_aptos", exist_ok=True)
parquet_url = "https://huggingface.co/datasets/bumbledeep/aptos/resolve/main/train.parquet"
dest = "data/raw/hf_aptos/train.parquet"
if not os.path.exists(dest):
    print("Downloading APTOS parquet dataset...")
    urllib.request.urlretrieve(parquet_url, dest)
    print("Downloaded successfully.")

# Unpack images and build quality-audited manifest
!python scripts/prepare_hf_aptos.py --parquet data/raw/hf_aptos/train.parquet --output-root data/raw/aptos
!python scripts/build_dataset_manifest.py
!python scripts/audit_manifest_quality.py

In [ ]:
# Step 4: Run the 8-Stage Experimental Progression Benchmark
!python scripts/run_experiments_pipeline.py --output-dir artifacts

In [ ]:
# Step 5: Full Multi-Stage GPU Training (Exp 4-6)
!python scripts/train_manifest.py \
  --manifest data/processed/manifest_quality_accepted.csv \
  --output-dir artifacts/final_model \
  --model efficientnet_b0 \
  --epochs 25 \
  --batch-size 32 \
  --image-size 384 \
  --learning-rate 0.0002 \
  --label-smoothing 0.05 \
  --gradient-clip 1.0 \
  --patience 6

In [ ]:
# Step 6: Comprehensive Clinical & Robustness Evaluation
!python scripts/comprehensive_evaluation.py \
  --checkpoint artifacts/final_model/best_model.pt \
  --manifest data/processed/manifest_quality_accepted.csv \
  --image-size 384 \
  --output-dir artifacts/final_model

In [ ]:
# Step 7: Export Production Bundle to Local Machine
from google.colab import files
import os
for path in ["artifacts/final_model/best_model.pt", "artifacts/final_model/model_bundle.json", "artifacts/experiment_progression.json"]:
    if os.path.exists(path):
        print(f"Downloading {path}...")
        files.download(path)
print("Training & Export Complete!")